# 🧠 Clasificación: EDA + Balanceo + RandomForest, XGBoost, MLP, Keras y PyTorch

**Objetivo educativo:** dominar el flujo completo de un problema de **clasificación binaria** con clases desbalanceadas.

Usaremos el dataset **Breast Cancer Wisconsin** (maligno/benigno). Para hacer el problema más ilustrativo, **induciremos un desbalance artificial** para que los estudiantes vean el efecto de **SMOTE**.

### Cubriremos:
1. **EDA** — exploración, distribuciones, correlaciones y balance de clases.
2. **Detección de outliers** — método IQR.
3. **Balanceo con SMOTE** — comparación antes/después.
4. **Modelos** — Random Forest, XGBoost, MLP (sklearn), Keras y PyTorch.
5. **Curvas de entrenamiento** — para diagnosticar sobreajuste/subajuste.
6. **Métricas** — Accuracy, Precision, Recall, F1 y **AUC-ROC comparativo**.
7. **Comparación de complejidad**.


## 1. Importación de librerías

In [ ]:
# Librerías generales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# XGBoost
from xgboost import XGBClassifier

# Balanceo
from imblearn.over_sampling import SMOTE

# Keras / TensorFlow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
torch.manual_seed(SEED)

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 90
print("✅ Librerías cargadas correctamente")

## 2. Carga y desbalanceo del dataset

Tomaremos **Breast Cancer Wisconsin** y reduciremos artificialmente la clase minoritaria para simular un caso real de **desbalance**.

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame

# Renombramos para claridad
df.rename(columns={'target': 'clase'}, inplace=True)
# En sklearn: 0 = maligno, 1 = benigno. Vamos a INVERTIR para que 1 = maligno (positivo)
df['clase'] = 1 - df['clase']

print(f"Forma: {df.shape}")
print("\nBalance ORIGINAL:")
print(df['clase'].value_counts())
print(df['clase'].value_counts(normalize=True).round(3))

In [ ]:
# INDUCIMOS DESBALANCE: reducimos la clase 1 (maligno) al 15% de sus registros
np.random.seed(SEED)
clase_0 = df[df['clase'] == 0]
clase_1 = df[df['clase'] == 1].sample(frac=0.20, random_state=SEED)
df = pd.concat([clase_0, clase_1]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Balance INDUCIDO (dataset desbalanceado):")
print(df['clase'].value_counts())
print(df['clase'].value_counts(normalize=True).round(3))
df.head()

## 3. Análisis Exploratorio de Datos (EDA)

In [ ]:
df.info()

In [ ]:
df.describe().T.head(10)

In [ ]:
# Distribución de clases
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = df['clase'].value_counts().sort_index()
labels = ['Benigno (0)', 'Maligno (1)']

axes[0].bar(labels, counts.values, color=['steelblue', 'crimson'])
axes[0].set_title('Distribución de clases (frecuencia)')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%',
            colors=['steelblue', 'crimson'], startangle=90)
axes[1].set_title('Distribución de clases (proporción)')
plt.tight_layout(); plt.show()

razon = counts.max() / counts.min()
print(f"⚠️ Razón de desbalance: {razon:.2f} : 1")

In [ ]:
# Seleccionamos algunas variables clave para el EDA visual
variables_eda = ['mean radius', 'mean texture', 'mean perimeter', 'mean area',
                 'mean smoothness', 'mean concavity']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), variables_eda):
    for clase, color in zip([0, 1], ['steelblue', 'crimson']):
        ax.hist(df[df['clase']==clase][col], bins=25, alpha=0.6,
                label=f'Clase {clase}', color=color, density=True)
    ax.set_title(col); ax.legend()
plt.suptitle('Distribución de variables por clase', y=1.02, fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Boxplots por clase
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), variables_eda):
    sns.boxplot(x='clase', y=col, data=df, ax=ax, palette=['steelblue', 'crimson'])
    ax.set_title(col)
plt.suptitle('Boxplots por clase', y=1.02, fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Matriz de correlación (solo variables 'mean' para no saturar)
mean_cols = [c for c in df.columns if 'mean' in c] + ['clase']
plt.figure(figsize=(11, 8))
corr = df[mean_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', center=0, square=True, cbar_kws={'shrink': .8})
plt.title('Matriz de correlación (variables mean)')
plt.tight_layout(); plt.show()

## 4. Detección de outliers (método IQR)

In [ ]:
def detectar_outliers_iqr(df, columnas):
    resumen = {}
    for col in columnas:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lim_inf = Q1 - 1.5 * IQR
        lim_sup = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lim_inf) | (df[col] > lim_sup)]
        resumen[col] = {'n_outliers': len(outliers),
                        'porcentaje': round(100 * len(outliers) / len(df), 2)}
    return pd.DataFrame(resumen).T

columnas_num = df.columns.drop('clase')
outliers_df = detectar_outliers_iqr(df, columnas_num)
outliers_df.sort_values('n_outliers', ascending=False).head(10)

In [ ]:
# Winsorización de outliers (cortando a los límites IQR)
df_limpio = df.copy()
for col in columnas_num:
    Q1 = df_limpio[col].quantile(0.25)
    Q3 = df_limpio[col].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    df_limpio[col] = df_limpio[col].clip(lower=lim_inf, upper=lim_sup)

# Comparación visual (antes vs después)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=df[variables_eda], ax=axes[0], palette='Set2')
axes[0].set_title('ANTES de winsorización'); axes[0].tick_params(axis='x', rotation=30)
sns.boxplot(data=df_limpio[variables_eda], ax=axes[1], palette='Set2')
axes[1].set_title('DESPUÉS de winsorización'); axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## 5. Preprocesamiento: split, escalado y balanceo con SMOTE

**Regla de oro:** el balanceo se hace **solo sobre el train**, nunca sobre el test.

In [ ]:
X = df_limpio.drop(columns='clase').values
y = df_limpio['clase'].values

# División estratificada (mantiene proporción de clases)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)

# Escalado (importante para MLP, Keras, PyTorch)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"\nBalance train ANTES de SMOTE:")
print(pd.Series(y_train).value_counts())

In [ ]:
# Aplicamos SMOTE SOLO al train
smote = SMOTE(random_state=SEED)
X_train_bal, y_train_bal = smote.fit_resample(X_train_sc, y_train)

print(f"Balance train DESPUÉS de SMOTE:")
print(pd.Series(y_train_bal).value_counts())

# También guardamos versión no escalada balanceada para modelos como RF/XGB
X_train_bal_raw, y_train_bal_raw = SMOTE(random_state=SEED).fit_resample(X_train, y_train)

# Comparativa visual
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
pd.Series(y_train).value_counts().sort_index().plot.bar(
    ax=axes[0], color=['steelblue','crimson']
)
axes[0].set_title('ANTES de SMOTE'); axes[0].set_xticklabels(['Benigno','Maligno'], rotation=0)

pd.Series(y_train_bal).value_counts().sort_index().plot.bar(
    ax=axes[1], color=['steelblue','crimson']
)
axes[1].set_title('DESPUÉS de SMOTE'); axes[1].set_xticklabels(['Benigno','Maligno'], rotation=0)
plt.tight_layout(); plt.show()

### 📚 Alternativas al SMOTE
| Técnica | Descripción |
|---|---|
| **Undersampling** | Reducir la clase mayoritaria |
| **Oversampling simple** | Duplicar registros de la minoritaria |
| **SMOTE** | Generar sintéticos por interpolación |
| **ADASYN** | Similar a SMOTE, pero enfocado en muestras difíciles |
| **class_weight** | Penalizar más los errores en la minoritaria |


## 6. Función de evaluación

In [ ]:
def evaluar_clasif(nombre, y_train_true, y_train_pred, y_train_proba,
                   y_test_true, y_test_pred, y_test_proba, t_entreno=None):
    resultados = {
        'Modelo': nombre,
        'Acc_train':   accuracy_score(y_train_true, y_train_pred),
        'Acc_test':    accuracy_score(y_test_true, y_test_pred),
        'Precision':   precision_score(y_test_true, y_test_pred),
        'Recall':      recall_score(y_test_true, y_test_pred),
        'F1':          f1_score(y_test_true, y_test_pred),
        'AUC':         roc_auc_score(y_test_true, y_test_proba),
    }
    if t_entreno is not None:
        resultados['Tiempo (s)'] = round(t_entreno, 2)
    return resultados

resultados_globales = []
historias = {}     # curvas de aprendizaje
roc_data  = {}     # para el ROC comparativo

## 7. Modelo 1 · Random Forest

In [ ]:
t0 = time.time()
rf = RandomForestClassifier(n_estimators=200, max_depth=10,
                             random_state=SEED, n_jobs=-1)
rf.fit(X_train_bal_raw, y_train_bal_raw)  # RF no necesita escalado
t_rf = time.time() - t0

y_tr_pred  = rf.predict(X_train)
y_te_pred  = rf.predict(X_test)
y_te_proba = rf.predict_proba(X_test)[:, 1]
y_tr_proba = rf.predict_proba(X_train)[:, 1]

resultados_globales.append(
    evaluar_clasif('Random Forest', y_train, y_tr_pred, y_tr_proba,
                                     y_test, y_te_pred, y_te_proba, t_rf)
)
roc_data['Random Forest'] = (y_test, y_te_proba)
print(f"⏱ {t_rf:.2f}s  |  Acc test: {accuracy_score(y_test, y_te_pred):.3f}  |  AUC: {roc_auc_score(y_test, y_te_proba):.3f}")

In [ ]:
# Curva de aprendizaje
train_sizes, train_scores, val_scores = learning_curve(
    RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
    X_train_bal_raw, y_train_bal_raw,
    cv=3, scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 6), n_jobs=-1
)
historias['Random Forest'] = {
    'sizes': train_sizes,
    'train': train_scores.mean(axis=1),
    'val':   val_scores.mean(axis=1)
}

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Train', color='steelblue')
plt.plot(train_sizes, val_scores.mean(axis=1),   'o-', label='Validación', color='salmon')
plt.xlabel('Tamaño train'); plt.ylabel('Accuracy')
plt.title('Curva de aprendizaje — Random Forest')
plt.legend(); plt.grid(True); plt.show()

## 8. Modelo 2 · XGBoost

In [ ]:
t0 = time.time()
xgb = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    eval_metric='logloss',
    random_state=SEED, n_jobs=-1
)
xgb.fit(
    X_train_bal_raw, y_train_bal_raw,
    eval_set=[(X_train_bal_raw, y_train_bal_raw), (X_test, y_test)],
    verbose=False
)
t_xgb = time.time() - t0

y_tr_pred  = xgb.predict(X_train)
y_te_pred  = xgb.predict(X_test)
y_te_proba = xgb.predict_proba(X_test)[:, 1]
y_tr_proba = xgb.predict_proba(X_train)[:, 1]

resultados_globales.append(
    evaluar_clasif('XGBoost', y_train, y_tr_pred, y_tr_proba,
                              y_test, y_te_pred, y_te_proba, t_xgb)
)
roc_data['XGBoost'] = (y_test, y_te_proba)
print(f"⏱ {t_xgb:.2f}s  |  Acc test: {accuracy_score(y_test, y_te_pred):.3f}  |  AUC: {roc_auc_score(y_test, y_te_proba):.3f}")

In [ ]:
# Curvas por iteración (train vs test)
evals_result = xgb.evals_result()
epochs = range(len(evals_result['validation_0']['logloss']))

plt.figure(figsize=(8, 5))
plt.plot(epochs, evals_result['validation_0']['logloss'], label='Train', color='steelblue')
plt.plot(epochs, evals_result['validation_1']['logloss'], label='Test',  color='salmon')
plt.xlabel('Iteración (árboles)'); plt.ylabel('LogLoss')
plt.title('Curvas de entrenamiento — XGBoost')
plt.legend(); plt.grid(True); plt.show()

historias['XGBoost'] = {
    'sizes': list(epochs),
    'train': evals_result['validation_0']['logloss'],
    'val':   evals_result['validation_1']['logloss']
}

## 9. Modelo 3 · MLP (Perceptrón multicapa · sklearn)

In [ ]:
t0 = time.time()
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu', solver='adam',
    max_iter=200, random_state=SEED,
    early_stopping=True, validation_fraction=0.15
)
mlp.fit(X_train_bal, y_train_bal)
t_mlp = time.time() - t0

y_tr_pred  = mlp.predict(X_train_sc)
y_te_pred  = mlp.predict(X_test_sc)
y_te_proba = mlp.predict_proba(X_test_sc)[:, 1]
y_tr_proba = mlp.predict_proba(X_train_sc)[:, 1]

resultados_globales.append(
    evaluar_clasif('MLP (sklearn)', y_train, y_tr_pred, y_tr_proba,
                                     y_test, y_te_pred, y_te_proba, t_mlp)
)
roc_data['MLP (sklearn)'] = (y_test, y_te_proba)
print(f"⏱ {t_mlp:.2f}s  |  Acc test: {accuracy_score(y_test, y_te_pred):.3f}  |  AUC: {roc_auc_score(y_test, y_te_proba):.3f}")

# Curva de pérdida
plt.figure(figsize=(8, 5))
plt.plot(mlp.loss_curve_, color='steelblue', label='Pérdida train')
if hasattr(mlp, 'validation_scores_') and mlp.validation_scores_:
    plt.plot(mlp.validation_scores_, color='salmon', label='Score val')
plt.xlabel('Iteración'); plt.ylabel('Loss / Score')
plt.title('Curva de entrenamiento — MLP sklearn')
plt.legend(); plt.grid(True); plt.show()

## 10. Modelo 4 · Red neuronal con Keras (TensorFlow)

In [ ]:
def crear_modelo_keras(n_features):
    model = Sequential([
        Dense(64, activation='relu', input_shape=(n_features,)),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')   # sigmoide para binaria
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    return model

model_keras = crear_modelo_keras(X_train_bal.shape[1])
model_keras.summary()

In [ ]:
t0 = time.time()
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
hist_keras = model_keras.fit(
    X_train_bal, y_train_bal,
    validation_split=0.2,
    epochs=100, batch_size=32,
    callbacks=[early_stop], verbose=0
)
t_keras = time.time() - t0

y_tr_proba = model_keras.predict(X_train_sc, verbose=0).ravel()
y_te_proba = model_keras.predict(X_test_sc,  verbose=0).ravel()
y_tr_pred  = (y_tr_proba > 0.5).astype(int)
y_te_pred  = (y_te_proba > 0.5).astype(int)

resultados_globales.append(
    evaluar_clasif('Keras NN', y_train, y_tr_pred, y_tr_proba,
                                y_test, y_te_pred, y_te_proba, t_keras)
)
roc_data['Keras NN'] = (y_test, y_te_proba)
print(f"⏱ {t_keras:.2f}s ({len(hist_keras.history['loss'])} épocas)  |  Acc test: {accuracy_score(y_test, y_te_pred):.3f}  |  AUC: {roc_auc_score(y_test, y_te_proba):.3f}")

In [ ]:
# Curvas de entrenamiento Keras
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(hist_keras.history['loss'],     label='Train', color='steelblue')
axes[0].plot(hist_keras.history['val_loss'], label='Val',   color='salmon')
axes[0].set_title('Loss'); axes[0].set_xlabel('Época'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(hist_keras.history['accuracy'],     label='Train', color='steelblue')
axes[1].plot(hist_keras.history['val_accuracy'], label='Val',   color='salmon')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Época'); axes[1].legend(); axes[1].grid(True)

axes[2].plot(hist_keras.history['auc'],     label='Train', color='steelblue')
axes[2].plot(hist_keras.history['val_auc'], label='Val',   color='salmon')
axes[2].set_title('AUC'); axes[2].set_xlabel('Época'); axes[2].legend(); axes[2].grid(True)
plt.tight_layout(); plt.show()

historias['Keras NN'] = {
    'sizes': list(range(len(hist_keras.history['accuracy']))),
    'train': hist_keras.history['accuracy'],
    'val':   hist_keras.history['val_accuracy']
}

## 11. Modelo 5 · Red neuronal con PyTorch

Bucle de entrenamiento manual — ideal para que los estudiantes vean cómo funciona por dentro.

In [ ]:
# División interna para validación
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_bal, y_train_bal, test_size=0.2, random_state=SEED, stratify=y_train_bal
)

X_tr_t  = torch.tensor(X_tr,  dtype=torch.float32)
y_tr_t  = torch.tensor(y_tr,  dtype=torch.float32).view(-1, 1)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
X_te_t  = torch.tensor(X_test_sc, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True)

class MLPClasifTorch(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),         nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 1),          nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model_torch = MLPClasifTorch(X_tr_t.shape[1])
optim  = torch.optim.Adam(model_torch.parameters(), lr=1e-3)
criter = nn.BCELoss()
print(model_torch)

In [ ]:
t0 = time.time()
n_epocas = 60
hist_torch = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoca in range(n_epocas):
    model_torch.train()
    epoch_loss = 0.0
    n_correct = 0
    for xb, yb in train_loader:
        optim.zero_grad()
        pred = model_torch(xb)
        loss = criter(pred, yb)
        loss.backward()
        optim.step()
        epoch_loss += loss.item() * xb.size(0)
        n_correct  += ((pred > 0.5).float() == yb).sum().item()
    train_loss = epoch_loss / len(train_loader.dataset)
    train_acc  = n_correct  / len(train_loader.dataset)
    
    model_torch.eval()
    with torch.no_grad():
        val_pred = model_torch(X_val_t)
        val_loss = criter(val_pred, y_val_t).item()
        val_acc  = ((val_pred > 0.5).float() == y_val_t).float().mean().item()
    
    hist_torch['train_loss'].append(train_loss)
    hist_torch['val_loss'].append(val_loss)
    hist_torch['train_acc'].append(train_acc)
    hist_torch['val_acc'].append(val_acc)
    
    if (epoca + 1) % 10 == 0:
        print(f"Época {epoca+1:3d} | Train loss: {train_loss:.4f} acc: {train_acc:.3f} | Val loss: {val_loss:.4f} acc: {val_acc:.3f}")

t_torch = time.time() - t0

# Predicciones finales
model_torch.eval()
with torch.no_grad():
    y_tr_proba = model_torch(torch.tensor(X_train_sc, dtype=torch.float32)).numpy().ravel()
    y_te_proba = model_torch(X_te_t).numpy().ravel()

y_tr_pred = (y_tr_proba > 0.5).astype(int)
y_te_pred = (y_te_proba > 0.5).astype(int)

resultados_globales.append(
    evaluar_clasif('PyTorch NN', y_train, y_tr_pred, y_tr_proba,
                                  y_test, y_te_pred, y_te_proba, t_torch)
)
roc_data['PyTorch NN'] = (y_test, y_te_proba)
print(f"\n⏱ Total: {t_torch:.2f}s  |  Acc test: {accuracy_score(y_test, y_te_pred):.3f}  |  AUC: {roc_auc_score(y_test, y_te_proba):.3f}")

In [ ]:
# Curvas de entrenamiento PyTorch
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist_torch['train_loss'], label='Train', color='steelblue')
axes[0].plot(hist_torch['val_loss'],   label='Val',   color='salmon')
axes[0].set_title('Loss'); axes[0].set_xlabel('Época'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(hist_torch['train_acc'], label='Train', color='steelblue')
axes[1].plot(hist_torch['val_acc'],   label='Val',   color='salmon')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Época'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()

historias['PyTorch NN'] = {
    'sizes': list(range(len(hist_torch['train_acc']))),
    'train': hist_torch['train_acc'],
    'val':   hist_torch['val_acc']
}

## 12. Comparativa de métricas

In [ ]:
df_resultados = pd.DataFrame(resultados_globales).set_index('Modelo')
df_resultados.round(4)

In [ ]:
# Diagnóstico de sobreajuste/subajuste
df_resultados['Gap Acc (train-test)'] = df_resultados['Acc_train'] - df_resultados['Acc_test']

def diagnostico(row):
    if row['Acc_train'] < 0.7 and row['Acc_test'] < 0.7:
        return '⚠️ Subajuste'
    elif row['Gap Acc (train-test)'] > 0.1:
        return '⚠️ Sobreajuste'
    else:
        return '✅ Buen balance'

df_resultados['Diagnóstico'] = df_resultados.apply(diagnostico, axis=1)
df_resultados[['Acc_train','Acc_test','Gap Acc (train-test)','Diagnóstico']].round(3)

In [ ]:
# Gráfico de barras comparativo
metricas = ['Acc_test', 'Precision', 'Recall', 'F1', 'AUC']
df_plot = df_resultados[metricas]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(df_plot))
w = 0.15
colores = ['steelblue', 'salmon', 'lightgreen', 'gold', 'purple']
for i, m in enumerate(metricas):
    ax.bar(x + i*w, df_plot[m], w, label=m, color=colores[i], alpha=0.85)

ax.set_xticks(x + 2*w)
ax.set_xticklabels(df_plot.index, rotation=15)
ax.set_ylabel('Puntaje'); ax.set_ylim(0, 1.05)
ax.set_title('Comparativa de métricas de clasificación')
ax.legend(loc='lower right'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 13. Curvas ROC comparativas

La **curva ROC** representa el trade-off entre **True Positive Rate** (recall) y **False Positive Rate**. Cuanto más cerca de la esquina superior izquierda, mejor. El **AUC** (Area Under the Curve) resume la calidad del modelo en un solo número.

In [ ]:
plt.figure(figsize=(9, 7))
for nombre, (y_true, y_proba) in roc_data.items():
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc = roc_auc_score(y_true, y_proba)
    plt.plot(fpr, tpr, linewidth=2.2, label=f'{nombre}  (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Clasificador aleatorio')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Curvas ROC comparativas')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True); plt.tight_layout(); plt.show()

## 14. Matrices de confusión de todos los modelos

In [ ]:
# Recolectamos predicciones de cada modelo para las matrices
predicciones_test = {
    'Random Forest':  rf.predict(X_test),
    'XGBoost':        xgb.predict(X_test),
    'MLP (sklearn)':  mlp.predict(X_test_sc),
    'Keras NN':       (model_keras.predict(X_test_sc, verbose=0).ravel() > 0.5).astype(int),
    'PyTorch NN':     (model_torch(X_te_t).detach().numpy().ravel() > 0.5).astype(int)
}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
for i, (nombre, y_pred) in enumerate(predicciones_test.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Benigno', 'Maligno'],
                yticklabels=['Benigno', 'Maligno'], cbar=False)
    axes[i].set_title(nombre)
    axes[i].set_xlabel('Predicho'); axes[i].set_ylabel('Real')

axes[-1].axis('off')  # celda sobrante
plt.tight_layout(); plt.show()

## 15. Curvas de entrenamiento comparativas

In [ ]:
plt.figure(figsize=(11, 6))
for nombre, h in historias.items():
    x = np.linspace(0, 1, len(h['train']))
    plt.plot(x, h['train'], '--', alpha=0.5, label=f'{nombre} — Train')
    plt.plot(x, h['val'],   '-',  label=f'{nombre} — Val/Test')
plt.xlabel('Progreso del entrenamiento (normalizado)')
plt.ylabel('Métrica (Accuracy o Loss según el modelo)')
plt.title('Curvas de entrenamiento comparativas')
plt.legend(fontsize=8, loc='best'); plt.grid(True)
plt.tight_layout(); plt.show()

## 16. Comparación de complejidad

In [ ]:
params = {
    'Random Forest': sum(t.tree_.node_count for t in rf.estimators_),
    'XGBoost':       xgb.get_booster().trees_to_dataframe().shape[0],
    'MLP (sklearn)': sum(c.size for c in mlp.coefs_) + sum(b.size for b in mlp.intercepts_),
    'Keras NN':      model_keras.count_params(),
    'PyTorch NN':    sum(p.numel() for p in model_torch.parameters())
}

df_complejidad = pd.DataFrame({
    'Parámetros / Nodos': params,
    'Tiempo entreno (s)': df_resultados['Tiempo (s)'],
    'AUC test': df_resultados['AUC'].round(3)
})
df_complejidad

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

x = np.arange(len(df_complejidad))
ax1.bar(x, df_complejidad['Parámetros / Nodos'], color='steelblue', alpha=0.6, label='Parámetros / Nodos')
ax1.set_yscale('log'); ax1.set_ylabel('Parámetros / Nodos (log)', color='steelblue')
ax1.set_xticks(x); ax1.set_xticklabels(df_complejidad.index, rotation=30)

ax2.plot(x, df_complejidad['AUC test'], 'o-', color='crimson', linewidth=2, markersize=10, label='AUC test')
ax2.set_ylabel('AUC test', color='crimson')
ax2.set_ylim(0.8, 1.02)

plt.title('Complejidad del modelo vs. AUC en test')
plt.tight_layout(); plt.show()

## 17. Conclusiones

### Preguntas para reflexión con los estudiantes

1. **¿Cuál sería la mejor métrica si estuviéramos detectando tumores malignos: Precision o Recall?** ¿Por qué?
2. **Sin SMOTE, ¿qué habría pasado con el Recall de la clase minoritaria?** (Experimenta desactivándolo).
3. **¿Qué modelo tiene mejor AUC?** ¿Coincide con el que tiene mejor Accuracy?
4. **¿Hay algún modelo con sobreajuste evidente?** Pista: revisa `Gap Acc` y las curvas de entrenamiento.
5. **¿Vale la pena la complejidad de las redes neuronales en este problema?** Compara con Random Forest en el gráfico de complejidad vs AUC.
6. **¿Cómo cambiaría el análisis si en lugar de SMOTE usáramos `class_weight='balanced'`?**

### 📚 Notas didácticas
- **Accuracy** engaña con clases muy desbalanceadas: un modelo que siempre diga "clase mayoritaria" puede tener 90% de accuracy y ser inútil.
- **Precision** = de los que predije positivos, ¿cuántos lo son realmente? (Evita falsos positivos)
- **Recall** = de los positivos reales, ¿cuántos detecté? (Evita falsos negativos — crítico en medicina)
- **F1** es la media armónica de Precision y Recall — útil cuando importan ambos.
- **AUC-ROC** es independiente del threshold — muestra la capacidad discriminativa global.
- El **gap train − test** en cualquier métrica es el mejor indicador de sobreajuste.

### 🧪 Ejercicios propuestos
1. Repite el análisis desactivando SMOTE y compara.
2. Cambia el hiperparámetro `max_depth` de RandomForest a 2 y a 30. Observa el impacto.
3. Añade regularización L2 al MLP de Keras y observa las curvas.
4. Prueba con `class_weight='balanced'` en RF y XGBoost como alternativa a SMOTE.
